# Practical 4: Pandas data discovery - Raster vs. Vector Data 

<div class="alert alert-block alert-success">
<b>Objectives:</b> In this practical we will use the GeoPandas package to load data from files and visualise our results on a map. We will introduce the concepts of both raster and vector data, and how these underline the approach of <b>Geographic Information Systems, or GIS</b>. To do this we will open and manipulate datafiles that contains information on emissions and model estimates across the UK. We will do this through the following activities [starting with social data!]:
    
 - 1) [Introduction: Raster vs. Vector Data in Environmental Science](#Part1)
      * [Exercise 1: Create a map showing social deprivation indices across Manchester](#Exercise1)
 - 2) [Adding vector point data and combining GeoPandas dataframes](#Part2)
      * [Exercise 2: Exercise 2: Compare the distribution of IMD Deciles at AURN sites that are categorised as either 'Urban Traffic' or 'Suburban Background'](#Exercise2)
 - 3) [Importing emissions maps from the National Emissions Inventory](#Part3)
      * [Exercise 3: Create a map of PM2.5 emissions across Greater Manchester](#Exercise3)
 - 4) [Calculating distances between points and polygons](#Part3)
      * [Exercise 4: Create a map of PM2.5 emissions in a single ward in Greater Manchester 10 years apart and calculate the difference in totals.](#Exercise4)

As with our other notebooks, we will provide you with a template for plotting the results. Also please note that you should not feel pressured to complete every exercise in class. These practicals are designed for you to take outside of class and continue working on them. Proposed solutions to all exercises can be found in the 'Solutions' folder.

Please do not worry about trying to remember all of the syntax in the code we provide. As noted in the lectures, this course is aimed at building experience in using Python and many of its features. Even seasoned professionals have to look up solutions from old files or online! 
</div>


<div style="border-left: 6px solid #1f77b4; background-color: #f0f8ff; padding: 15px; margin: 15px 0;">

### ✅ <span style="color:#1f77b4">Key Learning Outcomes – Practical 4: Pandas Data Discovery with Raster and Vector Data</span>

In this practical session, you will explore the foundations of **Geographic Information Systems (GIS)** by working with both **vector** and **raster spatial data** in Python. Using real environmental and social datasets, you'll practice reading, visualising, and analyzing geospatial patterns in air pollution, emissions, and inequality across Greater Manchester.

#### 🧠 Learning Objectives:

1. **Understand the Difference Between Raster and Vector Data**
   - Learn how spatial data is structured and used in GIS.
   - Identify when to use **vector shapes** (e.g. boundaries, points) vs. **raster surfaces** (e.g. emissions maps).

2. **Visualise Social and Environmental Indicators Using GeoPandas**
   - Plot **Index of Multiple Deprivation (IMD)** deciles across Manchester using choropleth mapping.
   - Add **point data** for air quality monitoring stations (AURN) and explore spatial joins.

3. **Overlay and Compare Multiple Spatial Datasets**
   - Combine vector and raster layers in the same plot.
   - Compare deprivation levels at different **site types** (e.g. Urban Traffic vs. Suburban Background).

4. **Load and Display Raster Emissions Data**
   - Use `rasterio` to import and clip **PM₂.₅ emissions data** from the UK National Emissions Inventory.
   - Match raster layers to administrative boundaries and plot pollution gradients.

5. **Perform Spatial Distance Calculations**
   - Calculate the **distance between point locations and polygons** (e.g. AURN sites and LSOA areas).
   - Identify socially deprived areas located **within a buffer** of monitoring stations.

#### 📍 Additional Notes:
- You'll integrate **social inequality data** with **environmental emissions data** to explore real-world research questions.
- This practical builds skills in **spatial data wrangling**, **map design**, and **basic spatial analytics**, all crucial for environmental science, urban planning, and public health applications.

</div>

<div class="alert alert-block alert-danger">
<b> Using Google Colabs </b> Please note that the first code block ensures that we can access our files IF we are running this example on Google Colab. Please run this code block and, if you are not on Google Colab, you will recieve a message that confirms this. 
</div>

In [ ]:
if 'google.colab' in str(get_ipython()):
    !pip install contextily rasterio
else:
     print('Not running on CoLab')

if 'google.colab' in str(get_ipython()):
    from google.colab import drive
    mount='/content/gdrive'
    print("Colab: mounting Google drive on ", mount)
    drive.mount(mount)
    drive_root = mount + "/My Drive/Colab Notebooks/MPEC_bootcamp/Time-series-analytics-course"
    # Change to the directories to get data files
    print("\nColab: Changing directory to ", drive_root)
    %cd $drive_root
else:
    print('Not running on CoLab')

# 1) Introduction: Raster vs. Vector Data in Environmental Science  <a name="Part1">

In most areas of environmental science spatial data is pivotal for analysis and visualization; So much so that it is useful we briefly extend our geospatial learning to include more *formal* definitions of processes and data types. 

If you start to work more frequently with geospatial data, you will inevitably come across **Geographic Information Systems**, or more commonly referred to as **GIS**. GIS is a system used for capturing, storing, checking, and displaying data related to positions on Earth's surface. It helps analysts understand patterns, relationships, and geographic context. In our course so far we have structured our learning around types of data storage and ways to create and access data. When we refer to GIS we might consider this a seperate field by itself. Indeed, there are multiple components of GIS:

- **Data**: The core of GIS, encompassing geographical (spatial) data and attribute (descriptive) data.
- **Tools for Analysis**: GIS provides tools to analyze and interpret data, revealing patterns and relationships.
- **Visualization**: Beyond maps, GIS allows for sophisticated visual representations like 3D models and thematic maps.

How is this different from what we have looked at so far? GIS essentially is a system for:

- **Layering Information**: GIS layers different types of data (like roads, rivers, population) to show relationships and patterns.
- **Spatial Analysis**: It can perform complex spatial analyses, answering questions like "What is the shortest route?" or "What areas are at high risk for flooding?"

GIS is often, and incorrectly, thought to remain a tool solely within the field of Geography. GIS is not just for geographers; it's used in history, archaeology, environmental science, engineering, and more. Many use dedicated GIS software products to perform the above. The most common include: 

- **ArcGIS**: Developed by Esri, ArcGIS is one of the most widely used GIS software tools globally. It offers a suite of professional GIS applications, including ArcMap, ArcGIS Pro, and various online tools. It's known for its extensive capabilities in data visualization, analysis, and management.
- **QGIS (Quantum GIS)**: An open-source alternative to ArcGIS, QGIS is highly popular among users who prefer open-source. It offers a  set of features for mapping and spatial analysis. Its compatibility with various file formats and extensible nature through plugins makes it a versatile choice.
- **Google Earth Engine**: An advanced cloud-based tool for planetary-scale environmental data analysis. It's known for its massive archive of satellite imagery and geospatial datasets, which are useful for large-scale environmental monitoring and analysis.
- **....**

Tools vary widely in functionality, complexity, and cost, catering to different needs and skill levels. In this notebook we will find that there are modules, thus tools, in Python that can perform the aforementioned GIS functionality of layering through to spatial analysis. However we need to add new formal defintions to the core of GIS, the underlying data, before we begin. 

## Raster and Vector Data

Geospatial data primarily comes in two forms: **raster** and **vector** data. Understanding their differences, applications, and formats is important in the field. 

![](https://github.com/loftytopping/DEES_programming_course/blob/master/images/featured3.png?raw=true) 


### Raster Data

Raster data represents the Earth's surface as a matrix of cells, with each cell holding a value to represent a certain attribute like temperature or elevation. You can think of this as a checkerboard placed over the ground, where there is a value in each cell of the board. This may represent any variable of interest such as temperature, soil moisture, height and so on. The left hand side of figure 1 illustrates this concept. We can imagine opening a raster dataset in a wide range of software products, including Excel if we really wanted to.

### Vector Data

Vector data uses geometric shapes (points, lines, polygons) to represent spatial features like locations (wells, weather stations), paths (rivers, roads), and areas (lakes, political boundaries). We have already encoutered points in the previous notebook. Coordinates of single locations as a function of time resemble a raster dataset. 

Key characteristics and examples of both **raster** and **vector** data are given in the table below. You can also find common formats used for storing the data. That isnt to say we will stop using CSV files. However, we quickly find that CSV files are rather limited when we start to work with data that is more complex than tabular structures. CSV files are also relatively inefficient so larger quantities of data require bespoke formats. We have already noted that Pandas can work with a wide range of data formats. For example, JSON stands for JavaScript Object Notation. It is a lightweight data-interchange format that is easy for humans to read and write, and easy for machines to parse and generate. 


If you were to open a GEOJSON file in a text editor you might find something similar to the following example containing a single point feature:

```json
{
  "type": "FeatureCollection",
  "features": [
    {
      "type": "Feature",
      "geometry": {
        "type": "Point",
        "coordinates": [-73.9617, 40.6629]
      },
      "properties": {
        "name": "Brooklyn",
        "borough": "Kings",
        "state": "NY"
      }
    }
  ]
}
```

In this example, the GeoJSON object is a FeatureCollection. It contains an array of Feature objects. Each Feature has a geometry (in this case, a Point with coordinates) and properties (additional data about the feature). GeoJSON introduces specific types such as Point, LineString, Polygon, MultiPoint, MultiLineString, MultiPolygon, and GeometryCollection. Each of these types is used to represent different types of geographical features. We will be working with SHP files [see table below - Shapefiles] in this notebook. I cant actually provide you with an example of a SHP file per the GeoJOSN file. Shapefiles consist of several files that work together, and they contain binary data which is not human-readable in the same way.  They are however much more efficient in terms of storage requirements. The Shapefile format, commonly known as .shp, was created by Esri (Environmental Systems Research Institute), a leading company in the field of geographic information systems (GIS). Esri developed the Shapefile format in the early 1990s for their GIS software, and it quickly became a popular format for storing geographical data due to its simplicity and ease of use. 


| Data Type | Key Characteristics                                                                 | Common Formats                                  | Examples                                             |
|-----------|-------------------------------------------------------------------------------------|-------------------------------------------------|-----------------------------------------------------|
| Raster    | - Grid format (rows and columns) <br> - Cell values for attributes <br> - Resolution dependent | - GeoTIFF (.tif) <br> - ERDAS IMAGINE (.img) <br> - Digital Elevation Models (DEM) | - Satellite imagery <br> - Elevation data (DEMs) <br> - Land cover maps |
| Vector    | - Shape-based (points, lines, polygons) <br> - Precise location <br> - Attribute richness | - Shapefile (.shp) <br> - GeoJSON (.geojson) / TopoJSON (.topojson) <br> - KML/KMZ (.kml/.kmz) | - Map of geological sample locations <br> - River network diagrams <br> - Boundary maps for geological formations |

You may well ask what category our coordinate data from the previous notebook would fall within. That would be classed as *point vector* data since it was not provided under a fixed grid, but rather a set of latitude and longitude values for each eruption. In this notebook we will also work with polygon and line vector data. Choosing between raster and vector data depends on the data's nature and the analysis requirements. 

- **Analysis type**: Some analyses favor the continuous nature of raster data; others need the precision of vector data.
- **Data integration**: Often, integrating both raster and vector data is necessary.
- **Visualization**: Raster and vector formats differ significantly in mapping and visualization.

Understanding raster and vector data's differences, along with their specific formats, is essential for geoscientists to effectively analyze, integrate, and visualize spatial information.

One more point of discussion before we begin..

A slight *tweak* to Pandas is required to deal with line and polygon vector data. The good news is that someone has already done this for us. A module called GeoPandas is built exactly to deal with line and polygon data. 

## What is GeoPandas?

GeoPandas is an open-source project that makes working with geospatial data in Python easier. It extends the datatypes used by pandas to allow spatial operations on geometric types. Geometric operations are performed by `shapely`. GeoPandas further depends on `fiona` for file access and `matplotlib` for plotting. These are all modules that are installed when we install GeoPandas using the `conda` package manager. Figure xx below provides a simple illustration of GeoPandas. Features of GeoPandas include:

- **Geospatial Data Handling**: Simplifies handling of geospatial data (like shapefiles, GeoJSON) in Python.
- **Integration with pandas**: Builds upon and extends pandas, making it easy to work with geospatial data using familiar pandas functionalities.
- **Geometry Operations**: Enables geometry operations (like calculating area, distance, buffering) directly on GeoDataFrame.
- **Plotting and Visualization**: Offers straightforward plotting of geospatial data, which integrates well with `matplotlib`.

The GeoDataFrame is the core data structure in GeoPandas, which can store geometry columns and perform spatial operations.

![](https://github.com/loftytopping/DEES_programming_course/blob/master/images/geopandas.png?raw=true) 

notice how we still retain the concept of an index, but have a seperate column that can store shapes. We will load a `.shp` file that contains information on bedrock geology across the UK. First we will import relevant modules that we will use across this notebook. These are:

 - GeoPandas
 - Contextily [a small Python 3 (3.7 and above) package to retrieve tile maps from the internet](https://contextily.readthedocs.io/en/latest/). It can add those tiles as a basemap to matplotlib figures, which gives us the underlying map.
 - Matplotlib. As usual...this deals with all of our plotting requirements.

In [ ]:
from pathlib import Path
import geopandas as gpd
import contextily as cx
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import rasterio
from rasterio.plot import show
from rasterio.mask import mask as rio_mask
import matplotlib.colors as colors
from mpl_toolkits.axes_grid1 import make_axes_locatable
%matplotlib inline

DATA_DIR = Path('data')

# Load IMD shapefile
file = DATA_DIR / 'Lower_Super_Output_Area_IMD2019/Lower_Super_Output_Area_(LSOA)_IMD2019_(WGS84).shp'
IMD = gpd.read_file(file)
IMD

Notice how GeoPandas, like Pandas, has created its own indexing system of assigning a number to each row. There are also mulitple columns. Unfortunately you will often find it is difficult to obtain 'meta data' associated with records. Meta-data provides us with important information, such as when data was collected and so on. 

The original data has been downloaded from the following URL: https://communitiesopendata-communities.hub.arcgis.com/datasets/d473e9ad137240b6aa47c9e3f4bdd674_0/explore

For information, the English Indices of Deprivation 2019 use 39 separate indicators, organised across seven distinct domains of deprivation which can be combined, using appropriate weights, to calculate the Index of Multiple Deprivation 2019 (IMD 2019). This is an overall measure of multiple deprivation experienced by people living in an area and is calculated for every Lower layer Super Output Area (LSOA) in England. The IMD 2019 can be used to rank every LSOA in England according to their relative level of deprivation. Areas are ranked from the most deprived area (rank 1) to the least deprived area. Each nation publishes its data on its own data portal. Each nation measures deprivation in a slightly different way but the broad themes include income, employment, education, health, crime, barriers to housing and services, and the living environment.

In this instance, I want to create a map using the column **IMDDecil** which describes deprivation for every LSOA as a decile. A decile is a quantitative method of splitting up a set of ranked data into 10 equally large subsections. To place this on a map, I need coordinates.  If you scroll to the right hand edge if the dataframe you will find a column called `geometry`. This contains `POLYGON` with what appear to be a series of floating point numbers. This brings us on to an important check-point; Coordinate Reference Systems. 

## Coordinate Reference System (CRS) 

Before we create our map using the information provided in this file, we need to check what system is being used to represent the coordinates. If you take a look at the column `geometry` we can clearly see these do not represent latitude and longitude numbers. But why? A Coordinate Reference System (CRS) defines how the two-dimensional, projected map in your GeoDataFrame relates to real places on the earth. CRS can be represented in various formats, such as EPSG codes. EPSG codes are short identifiers assigned to coordinate reference system (CRS) definitions. These codes are standardized and maintained by the EPSG (European Petroleum Survey Group) Geodetic Parameter Dataset. EPSG codes cover both geographic coordinate systems (GCS) and projected coordinate systems (PCS). A GCS uses latitude and longitude to define locations on the earth's surface, while a PCS provides a flat, two-dimensional representation of the earth.

The following table lists some of the most frequently used EPSG codes in GIS, along with their descriptions:

| EPSG Code | CRS Name                   | Description                                                                                   |
|-----------|----------------------------|-----------------------------------------------------------------------------------------------|
| EPSG:4326 | WGS 84                     | Standard for global geographic coordinate system, used in GPS and for global datasets.        |
| EPSG:3857 | WGS 84 / Pseudo-Mercator   | Also known as Web Mercator. Common for web mapping services like Google Maps and OpenStreetMap. |

**Key Points**:
- **EPSG:4326** is widely used for datasets that span the entire globe. It represents coordinates in latitude and longitude.
- **EPSG:3857** is used for displaying maps on web applications, offering a compromise between a spherical and flat representation of the earth.
- ...
- **Using EPSG codes ensures that spatial data from different sources can be combined and analyzed accurately. Correct CRS selection (via EPSG codes) is crucial for accurate distance and area measurements, especially over large geographic extents. This is crucial when combining data from different sources to ensure they align correctly.**

So we need to check what Coordinate Reference System (CRS) our bedrock datafile is in. To do this, we can use the ```python .crs ``` property of our GeoDataframe as per the code box below.

In [ ]:
IMD.crs

We are told this data is stored using a projected coordinate system.For applications that require global positioning, such as GPS, data often needs to be transformed to a more universally recognized datum like EPSG:4326. We are going to convert all of our loaded vector and raster files into EPSG:3857. As we combine data from different sources, we need to ensure they are based on the same CRS. Fortunately, GeoPandas allows us to move between different systems. In the code snippet below, we create reference to a new GeoDataframe using the `.to_crs` function. We also preview it and notice that data in the geometry column has indeed changed. Notice this is a dummy function for now, since the data is already in the right coordinate system, but run this nonetheless.

In [ ]:
IMD_wm = IMD.to_crs(epsg=3857)
IMD_wm

Now we are close to producing our first map. GeoDataframes have a `.plot` function, like Pandas dataframes. By defauly they will plot the information stored within the geometry column. In the following short code snippet there are multiple operations and options provided. Lets break them down a little to try and understand a little more.

<div class="alert alert-block alert-info">
    
 - Create reference to a new axis, 'ax1', which is created when calling the `.plot` function. A number of plot options and arguments are provided. This include:
 > alpha - a level of transpareny for the IMD layers. I would like to see some of the underlying map
 > column - specify the column by which I want to classify the different polygons, thus label.
 > edgecolor - colour to seperate the polygons
 > cmap - the colour scheme to use for the different categories. There are many in Matplotlib: [https://matplotlib.org/stable/users/explain/colors/colormaps.html]
 > legend - do I want a legend? Yes.
 > legend_kwds - these are specific keywords that define the style and placing of the legend, which you do not need to remember!
    
 - Use contextily to add a basemap to the above axis. Contextily works by providing different styles of tiles to our figures. The default is to use OpenStreetMap, which looks like Google Maps. When we call the function `.add_basemap` we have to tell Contextily which axis to use, and the CRS system to map on to. It does all ofthe alignment for us! Note this is a web service so if not connected to the internet this may not work.

</div>

In [ ]:
# Create an axis [ax1] that has the information held within the gdf_wm dataframe. Provide styling and colouring option
ax1 = IMD_wm.plot(figsize=(16, 16), alpha=0.3, column="IMDDecil", edgecolor="k", cmap="gist_ncar",legend=True)
# Use the Contextily module to add a basemap underneath axes (ax1), which will look like OpenStreetMap. We make sure Contextily uses the same 'CRS' system by supplying this as an argument
cx.add_basemap(ax1, crs='EPSG:3857')
plt.show()

This is our first map.Its rather busy at the moment, so let's try to plot a subset:

## Selecting a subset of data. 

In our previous notebook we were able to select a subset of a dataframe. For example, if we only want to plot data from events that occured in 2019. To do this we can create a reduced dataframe created by the following code:

```python
data[data['Start Year'] == 2019]
```

I dont have any specific numeric criteria in this example. Rather, I want to select a subset according to the enries in the column **lsoa11nm**. The `.str.contains()` function in Pandas (and by extension, GeoPandas, since GeoPandas is an extension of Pandas) is used to check if each string in a Series or Index contains a specified pattern or substring. It's commonly used for filtering data based on string patterns. In our case, this means I can select a subset of our dataframe that has certain keywords. In the example below, I recreate the above map but only for entries that contain 'London' in the column **lsoa11nm**. Note that I first create a new GeoDataframe called `IMD_wm_new` based on this criteria. 

In [ ]:
IMD_wm_new = IMD_wm[IMD_wm["lsoa11nm"].str.contains('London')]
ax2 = IMD_wm_new.plot(figsize=(16, 16), alpha=0.3, column="IMDDecil", edgecolor="k", cmap="gist_ncar",legend=True);
cx.add_basemap(ax2, crs='EPSG:3857')
plt.show()

So we have a much smaller map over a restricted area of London. We are not investigating the social sciences in this noteboo, but our visualisation shows us there is a change in IMD status over a relatively short area. In the following exercise we ask you to do this for Manchester. Note that at the moment we are finding entries that match a substring. This isnt optimal, and shortly we will cut out the goespatial area we want using more vector data. Prior to the exercise, I provide some code that allows us to see which entries match the 'Manchester' requirement.

In [ ]:
# Define the pattern (e.g., words containing 'Manchester')
pattern = r'\bManchester\b'
# Filter rows where the pattern matches and get unique entries
unique_entries = IMD_wm[IMD_wm['lsoa11nm'].str.contains(pattern, regex=True, na=False)]['lsoa11nm'].unique()
print(unique_entries)

<div class="alert alert-block alert-success">
<b> Exercise 1: Create a map showing IMD status across Manchester. <a name="Exercise1"></a>  </b> I have chosen the name of a new dataframe for you.
</div>

<div style="border-left: 6px solid #ff8c00; background-color: #fff3e0; padding: 15px; margin: 15px 0;">

<details>
<summary><strong>🧪 Click to show solution</strong></summary>

```python

IMD_wm_new = IMD_wm[IMD_wm['lsoa11nm'].str.contains(pattern, regex=True, na=False)]
ax2 = IMD_wm_new.plot(figsize=(16, 16), alpha=0.3, column="IMDDecil", edgecolor="k", cmap="gist_ncar",legend=True);
cx.add_basemap(ax2, crs='EPSG:3857')
plt.show()

```

</details>

</div>

In [ ]:
#---------- INSERT CODE -------------------



#-------------------------------------------

# 2) Adding vector point data and combining GeoPandas dataframes   <a name="Part2">

## Adding vector point data

We now want to add a new layer to our map. GIS systems are built to add layers. To demonstrate this I am going to load in data that includes locations of the AURN network. **Do not worry about the syntax used..all we are interested in is extracting coordinates for each station and then creating another geo-dataframe**

In [ ]:
file = DATA_DIR / 'AURN/site_locations_with_postcode_211220.csv'
AURN_df = pd.read_csv(file)
AURN_gdf = gpd.GeoDataFrame(AURN_df, geometry=gpd.points_from_xy(AURN_df.longitude, AURN_df.latitude), crs="EPSG:4326")
AURN_gdf

Once again we have a **geomtry** column. However, now it is called a **POINT** rather than polygon. If we want to plot these points on our map, we again need to check the CRS system (Coordinate Reference System) used. If it dosnt match, we will need to convert. Lets again use the `.crs` function.

In [ ]:
AURN_gdf.crs

In [ ]:
AURN_gdf_wm = AURN_gdf.to_crs(epsg=3857)

How do we add this new data on to our existing national map? Recall in the first map exercise we created a new axis, **ax1**. This axis provides a canvas on which we can add datapoints. Indeed, we have already found that a dataframe as a `.plot` function. Using this function we can also specify which axis to plot on. Take a look at the new second line in the code snippet below. This looks very much like our previous Matplotlib plot functions. We also still end by using `Contextily` to add a basemap, ensuring we use the correct EPSG code.

In [ ]:
# Create a new axis for our bedrock map, using the same options as before.
ax1_new = IMD_wm.plot(figsize=(16, 16), alpha=0.3, column="IMDDecil", edgecolor="k", cmap="gist_ncar",legend=True)
# Now plot data from our new AURN dataframe, AURN_gdf, specifying the axis to host the datapoints.
AURN_gdf_wm.plot(ax=ax1_new, marker='o', color='red', markersize=5)
# Use contextily to add a basemap.
cx.add_basemap(ax1_new, crs='EPSG:3857')
plt.show()

Great. Now I want to create a map where only the IMDB and AURN location overlap. This is too complex to perform manually using a subset search. Fortunately we can join two different GeoDataframes and it will only overlap data where the geomtetry aligns.  We are going to join Pandas dataframes based on time information in a later practical, but we need overlapping geometries. 

## Join and Clip GeoPandas dataframes

The `.sjoin` (spatial join) function in GeoPandas is a tool for combining two GeoDataFrames based on their spatial relationship. It's akin to a standard join in Pandas, but instead of joining on keys, it joins based on spatial relationships between the geometries (like points, lines, polygons) in the GeoDataFrames. In our example, I can clip the AURN dataframe, which covers all of the UK, with the spatial area of the IMDB dataframe as follows

In [ ]:
# Spatial join
AURN_gdf_clipped=AURN_gdf_wm.clip(IMD_wm)
AURN_gdf_clipped

Now we have created this new dataframe, let's recreate our map. Notice how only a subset of both polygons and points are now displayed, but both align.

In [ ]:
# Create a new axis for our bedrock map, using the same options as before.
ax2_new = IMD_wm.plot(figsize=(16, 16), alpha=0.3, column="IMDDecil", edgecolor="k", cmap="gist_ncar",legend=True)
# Now plot data from our new AURN dataframe, AURN_gdf, specifying the axis to host the datapoints.
AURN_gdf_clipped.plot(ax=ax2_new, marker='o', color='red', markersize=5)
# Use contextily to add a basemap.
cx.add_basemap(ax2_new, crs='EPSG:3857')
plt.show()

How do we now investigate the IMD status of locations where AURN sites are based? For this we want to create a new dataframe where only the point and polygons. In a Spatial Join, two geometry objects are merged based on their spatial relationship to one another.`.sjoin` joins two dataframes based on a binary predicate performed on all combinations of geometries, one of `intersects`, `contains`, `within`, `touches`, `crosses`, or `overlaps`. You can specify whether you want a left, right, or inner join based on the how keyword argument.

In [ ]:
# Execute spatial join
IMD_with_AURN = gpd.sjoin(IMD_wm, AURN_gdf_clipped, how="inner")
IMD_with_AURN.head()

We can also now evaluate the distribution of IMD values within this subset. Specifically we we can apply the function `.value_counts()` to the column in our dataframe to then create a list as follows. 

In [ ]:
# Analyze distribution
distribution = IMD_with_AURN['IMDDecil'].value_counts()
# Print the distribution list
print(distribution)

To add another useful insight we can also create a histogram of the unique entries in our new dataframe, using the code below.

In [ ]:
# Create a histogram
plt.figure(figsize=(10, 6))  # You can adjust the size of the figure
distribution.plot(kind='bar')  # 'bar' for a bar chart (histogram)
plt.title('Distribution of Social Deprivation Deciles at AURN sites')
plt.xlabel('IMD Decile')
plt.ylabel('Frequency')
plt.xticks(rotation=90)  # Rotates the x labels to make them more readable
plt.show()

<div class="alert alert-block alert-success">
<b> Exercise 2: Compare the distribution of IMD Deciles at AURN sites that are categorised as either 'Urban Traffic' or 'Suburban Background'. <a name="Exercise1"></a>  

</b> I have chosen the name of a new dataframe for you. In the code below I have created a canvas with two subplots. Your job is to make sure the data being plotted in each one presents what is being asked.

</div>



<div style="border-left: 6px solid #ff8c00; background-color: #fff3e0; padding: 15px; margin: 15px 0;">

<details>
<summary><strong>🧪 Click to show solution</strong></summary>

```python
# Filter and compute value counts for each site type
suburban_dist = IMD_with_AURN[IMD_with_AURN['site_type'] == 'Suburban Background']['IMDDecil'].value_counts().sort_index()
urban_dist = IMD_with_AURN[IMD_with_AURN['site_type'] == 'Urban Background']['IMDDecil'].value_counts().sort_index()

# Ensure both Series have the same index (fill missing values with 0)
all_deciles = sorted(set(suburban_dist.index).union(set(urban_dist.index)))
suburban_dist = suburban_dist.reindex(all_deciles, fill_value=0)
urban_dist = urban_dist.reindex(all_deciles, fill_value=0)

# Create the plot
plt.figure(figsize=(10, 6))

# Plot both distributions with a bar width adjustment to avoid overlap
bar_width = 0.4
x = range(len(all_deciles))
plt.bar([i - bar_width/2 for i in x], suburban_dist, width=bar_width, label='Suburban Background')
plt.bar([i + bar_width/2 for i in x], urban_dist, width=bar_width, label='Urban Background')

# Customize plot
plt.title('Distribution of IMD Deciles at AURN Sites by Site Type')
plt.xlabel('IMD Decile')
plt.ylabel('Frequency')
plt.xticks(ticks=x, labels=all_deciles, rotation=90)
plt.legend()
plt.tight_layout()
plt.show()
```

</details>

</div>

In [ ]:
#---------- INSERT CODE -------------------



#-------------------------------------------

### Using GeoJSON vs Shapefiles for Regional Boundaries

Before we move on, it is important and useful to understand how we can also define regional boundaries on a map. For example, how do we display the boundary of Greater Manchester or The West Midlands combined authority?  Moreover, how can we use this information to 'cut out' data from geospatial files in the same way we use a cookie cutter. 

Both GeoJSON and ESRI Shapefiles store polygons (and multipolygons) plus attributes (names, codes, stats) to define areas like countries, counties or wards. As a bit of a recap:

#### GeoJSON
- **Format**: Plain JSON (RFC 7946), human-readable, ideal for web maps (Leaflet, Mapbox GL).  
- **CRS**: Defaults to WGS 84 (lon/lat).  
- **Use cases**:  
  - UK: ONS Open Geography, MapIt APIs stream boundaries as GeoJSON.  
  - Global: Natural Earth, GADM for quick prototyping.  


#### ESRI Shapefile
- **Components**: `.shp` (geometry) + `.shx` (index) + `.dbf` (attributes) + usually `.prj` (projection).  **stored in the same folder**
- **CRS**: Declared in `.prj` (e.g. EPSG:27700 for British National Grid, EPSG:4326 for WGS 84).  
- **Use cases**:  
  - UK: Ordnance Survey BoundaryLine in BNG.  
  - International: US TIGER/Line (geo coords).  
  - Desktop GIS (QGIS/ArcGIS), preserves field types and easy to edit.

In the code below I load a GeoJSON file for UK boundaries and plot the different wards within the Greater Manchester combined authority:
Please note in EPSG:3857 (Web Mercator), the coordinate axes are in metres (not degrees). So rather than “Longitude”/“Latitude” we label them Easting (m) and Northing (m)


In [ ]:
# Plot base map
fig, ax = plt.subplots(figsize=(10, 10))

file_GM = DATA_DIR / 'pollution/gmauthorities.geojson'
GMdf = gpd.read_file(file_GM)
GMdf_wm = GMdf.to_crs(epsg=3857)

GMdf_wm.plot(ax=ax, facecolor='none', edgecolor='blue')
cx.add_basemap(ax, crs='EPSG:3857')
plt.title("GMCA regional boundary")
plt.xlabel("Easting (m)")
plt.ylabel("Northing (m)")
plt.show()

Great, we can use this information to **clip** any geospatial data and then plot both on the same map. In the above we have both IMD data and locations of AURN sites. So lets produce a map showing all of these data within the regional boundaries.

In [ ]:
# Plot base map
fig, ax = plt.subplots(figsize=(10, 10))
divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="5%", pad=0.1)
IMD_wm_clipped = IMD_wm.clip(GMdf_wm)
AURN_gm_clipped = AURN_gdf_wm.clip(GMdf_wm)
GMdf_wm.plot(ax=ax, facecolor='none', edgecolor='blue')
IMD_wm_clipped.plot(ax=ax, alpha=0.3, column="IMDDecil", edgecolor="k", cmap="gist_ncar",
                    legend=True, cax=cax, legend_kwds={"orientation": "vertical"})
AURN_gm_clipped.plot(ax=ax, marker='o', color='red', markersize=5)
cx.add_basemap(ax, crs='EPSG:3857')
plt.title("GMCA regional boundary")
plt.xlabel("Easting (m)")
plt.ylabel("Northing (m)")
plt.show()

# 3) Importing emissions maps from the National Emissions Inventory   <a name="Part3">

## Working with Raster Data Using `rasterio` and GeoTIFF

You will often find that different air quality data assets require different sets of tools. Even when looking at geospatial data, we come across the need to load in and work with different dasta formats. 

The National Atmospheric Emissions Inventory (NAEI) is the official inventory of air pollutant emissions in the United Kingdom. It provides detailed estimates of the emissions of pollutants from a wide range of sources across the UK, and it is maintained to support government policy, public health, and environmental research. he NAEI is managed by Ricardo Energy & Environment on behalf of the UK Department for Energy Security and Net Zero (DESNZ) and the Department for Environment, Food and Rural Affairs (Defra). You can access the data and reports at the [official website](https://naei.beis.gov.uk). We have provided some data downloaded for you in the folder 'data' 

If you want to download the data, you will find it is possible to load geospatial data in the form of a GeoTiff. These are describe below.

---

### How Are GeoTIFFs Different from Shapefiles or GeoJSON?

| Format        | Type of Data | Description |
|---------------|--------------|-------------|
| **GeoTIFF**   | Raster       | Stores data in a grid of pixels. Ideal for continuous data like pollution or elevation. |
| **Shapefile** | Vector       | Represents features as points, lines, or polygons (e.g. roads, boundaries). |
| **GeoJSON**   | Vector       | A JSON-based format for encoding vector features. Easy to use with web tools. |

- **Raster data** is like an image with geolocation: each pixel has a value.
- **Vector data** is made up of geometric shapes (features) with associated attributes.

We cant easily load data in to a dataframe directly, just because of how a GeoTiff is constructed. We can however bring in another tool to get the data and then we can directly overlay this with data in our geodataframe. We will use a package called `rasterio`.

### What is `rasterio`?

[`rasterio`](https://rasterio.readthedocs.io/) is a Python library designed for working with **raster data**—gridded datasets where each pixel represents a geographic value (e.g. pollution levels, elevation, temperature). It provides powerful tools for reading, transforming, masking, and visualizing raster files like **GeoTIFFs**.

---


In the code below, we load in a geotif file that contains emission inventories for black carbon [named as totalbc in the filename]. You may be a little confused about the syntax used here. The rasterio package  opens a geotif file as a 'band' but also the transform and **coordinate reference system**. Run the code and you will find a very interesting looking map. Note that we have not provided any land boundaries from e.g. `contextily` - the features are simply related top the data being loaded in. Note also that, once the data is extracted, I convert this to a log10 base for easier viewing on the map. Can you identify potential sources from the features produced

In [ ]:
with rasterio.open(DATA_DIR / 'Pollution/totalbc22_2005.tif') as src:
    raster = src.read(1)
    transform = src.transform
    crs = src.crs

    raster_masked = np.ma.masked_where(raster <= 0, raster)
    raster_log = np.ma.log10(raster_masked)

    fig, ax = plt.subplots(figsize=(10, 10))
    show(raster_log, ax=ax, transform=transform, cmap='gist_rainbow', alpha=0.7)
    plt.title("Filtered Black Carbon Map (Values > 0)")
    plt.show()

How can we add this data on top of a map again? We can create a similar workflow to that we used before. In the code below, please note how I again make sure the **coordinate reference systems** between the data extracted from the tiff file and the contextily map are the same by making use of the `crs` we extracted from the tif file previously. In Matplotlib (and by extension in Contextily), zorder controls the drawing order of artists (plots, images, patches, etc.) on the axes—think of it like the “stack level” or “layer” each element lives on:

In [ ]:
with rasterio.open(DATA_DIR / 'Pollution/totalbc22_2005.tif') as src:
    arr = src.read(1)
    arr = np.ma.masked_where(arr <= 0, arr)
    arr = np.ma.log10(arr)
    transform = src.transform
    crs = src.crs

fig, ax = plt.subplots(figsize=(10, 10))
show(arr, transform=transform, ax=ax, cmap='gist_rainbow', alpha=0.7, zorder=2)
cx.add_basemap(ax, crs=crs, reset_extent=False, zorder=1)
ax.set_axis_off()
plt.tight_layout()
plt.show()

This looks nice, but I want to do some analysis with the data presented. In the code below I use `rasterio.mask` to clip the black carbon raster to our Greater Manchester boundary and overlay it on a map. **This assumes the Greater Manchester shapefile (`GMdf_wm`) is still in memory** — if not, re-run Cell 41.

## Step‐by‐Step Explanation of the BC‐Concentration Map Script

1. **Open the raster and clip to the boundary**
```python
with rasterio.open(DATA_DIR / 'Pollution/totalbc22_2005.tif') as src:
    GMdf_aligned = GMdf_wm.to_crs(src.crs)
    clipped_arr, clip_transform = rio_mask(src, GMdf_aligned.geometry, crop=True)
    crs = src.crs
```
Opens the GeoTIFF, reprojects `GMdf_wm` to match the raster's CRS, and clips the raster to those boundaries. `rio_mask` returns a 3D array (bands × rows × cols) and a new affine transform for the clipped area.

2. **Extract the first band**
```python
raster_clipped = clipped_arr[0]
transform = clip_transform
```
Our file has one band (band 0), so we index with `[0]` to get a 2D NumPy array.

3. **Mask out zeros and negatives**
```python
raster_masked = np.ma.masked_where(raster_clipped <= 0, raster_clipped)
```
Treats non-positive values as invalid so they won't appear in the plot.

4. **Apply log₁₀ normalisation for easier visualisation**
```python
raster_log = np.ma.log10(raster_masked)
```
Compresses the dynamic range for better visual contrast.

5. **Set up the figure and overlay the vector boundary**
```python
fig, ax = plt.subplots(figsize=(10, 10))
GMdf_aligned.plot(color='None', edgecolor='blue', linewidth=1, ax=ax, zorder=4)
cx.add_basemap(ax, crs=crs)
```
Draws the GM boundary outline on top of the basemap tiles.

6. **Compute the plotting extent**

When you plot a 2D array with `ax.imshow(...)`, Matplotlib assumes integer pixel indices by default. We need to tell it the real-world coordinates of the array corners using the affine transform:

```python
extent = [
    transform[2],                                          # xmin
    transform[2] + raster_log.shape[1] * transform[0],    # xmax
    transform[5] + raster_log.shape[0] * transform[4],    # ymin
    transform[5]                                           # ymax
]
```

7. **Render the raster and add a colorbar**
```python
im = ax.imshow(raster_log, cmap='gist_rainbow', extent=extent,
               alpha=0.3, origin='upper',
               norm=colors.Normalize(vmin=np.nanmin(raster_log),
                                     vmax=np.nanmax(raster_log)))
divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="5%", pad=0.05)
plt.colorbar(im, cax=cax).set_label("Log10(BC Concentration)")
```
Uses a rainbow colormap with semi-transparency so the basemap shows through.

In [ ]:
# Clip the black carbon raster to the Greater Manchester boundary
with rasterio.open(DATA_DIR / 'Pollution/totalbc22_2005.tif') as src:
    GMdf_aligned = GMdf_wm.to_crs(src.crs)
    clipped_arr, clip_transform = rio_mask(src, GMdf_aligned.geometry, crop=True)
    crs = src.crs

raster_clipped = clipped_arr[0]
transform = clip_transform

# Mask out zero values
raster_masked_clipped = np.ma.masked_where(raster_clipped <= 0, raster_clipped)

# Apply log10 transform
raster_log_clipped = np.ma.log10(raster_masked_clipped)

fig, ax = plt.subplots(figsize=(10, 10))

GMdf_aligned.plot(color='None', edgecolor='blue', linewidth=1, ax=ax, zorder=4)
cx.add_basemap(ax, crs=crs)

extent = [
    transform[2],
    transform[2] + raster_log_clipped.shape[1] * transform[0],
    transform[5] + raster_log_clipped.shape[0] * transform[4],
    transform[5]
]

im = ax.imshow(raster_log_clipped,
               cmap='gist_rainbow', extent=extent, alpha=0.3, origin='upper',
               norm=colors.Normalize(vmin=np.nanmin(raster_log_clipped),
                                     vmax=np.nanmax(raster_log_clipped)))

divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="5%", pad=0.05)
cbar = plt.colorbar(im, cax=cax)
cbar.set_label("Log10(BC Concentration)")

ax.set(title="2022 Log10(BC Concentration) emissions from NAEI")
plt.xlabel("Easting (m)")
plt.ylabel("Northing (m)")
plt.show()

<div class="alert alert-block alert-success">
<b> Exercise 3: Create a map of PM2.5 emissions across Greater Manchester. <a name="Exercise3"></a>  

</b> In this exercise we repeat the above code, but you will need to take a look at the data provided in the data folder and decide which to use. You will also need to decide whether to use log10 data or leave it in the range provided in the original file.

</div>


<div style="border-left: 6px solid #ff8c00; background-color: #fff3e0; padding: 15px; margin: 15px 0;">

<details>
<summary><strong>🧪 Click to show solution</strong></summary>

```python
with rasterio.open(DATA_DIR / 'Pollution/totalpm2_522_2022.tif') as src:
    GMdf_aligned = GMdf_wm.to_crs(src.crs)
    clipped_arr, clip_transform = rio_mask(src, GMdf_aligned.geometry, crop=True)
    crs = src.crs

raster_clipped = clipped_arr[0]
transform = clip_transform

# Mask out zero values
raster_masked_clipped = np.ma.masked_where(raster_clipped <= 0, raster_clipped)

# Apply log10 transform
raster_log_clipped = np.ma.log10(raster_masked_clipped)
```

</details>

</div>

In [ ]:
#---------- INSERT CODE -------------------
# Open the PM2.5 raster and clip to Greater Manchester
# Hint: use rasterio.open() and rio_mask() — same pattern as the black carbon example above

with rasterio.open(DATA_DIR / 'Pollution/') as src:
    GMdf_aligned = GMdf_wm.to_crs(src.crs)
    clipped_arr, clip_transform = rio_mask(src, GMdf_aligned.geometry, crop=True)
    crs = src.crs

raster_clipped = clipped_arr[0]
transform = clip_transform

# Mask out zero values
raster_masked_clipped = np.ma.masked_where(raster_clipped <= 0, raster_clipped)

# Apply log10 transform
raster_log_clipped =
#---------------------------------------

fig, ax = plt.subplots(figsize=(10, 10))

GMdf_aligned.plot(color='None', edgecolor='blue', linewidth=1, ax=ax, zorder=4)
cx.add_basemap(ax, crs=crs)

extent = [
    transform[2],
    transform[2] + raster_log_clipped.shape[1] * transform[0],
    transform[5] + raster_log_clipped.shape[0] * transform[4],
    transform[5]
]

im = ax.imshow(raster_log_clipped,
               cmap='gist_rainbow', extent=extent, alpha=0.3, origin='upper',
               norm=colors.Normalize(vmin=np.nanmin(raster_log_clipped),
                                     vmax=np.nanmax(raster_log_clipped)))

divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="5%", pad=0.05)
cbar = plt.colorbar(im, cax=cax)
cbar.set_label("Log10(PM2.5 Concentration)")

ax.set(title="2022 Log10(PM2.5 Concentration) emissions from NAEI")
plt.xlabel("Easting (m)")
plt.ylabel("Northing (m)")
plt.show()

# 4) Calculating distances  <a name="Part4">

In GeoPandas, the calculation of distances between points and polygons is carried out using the ```Shapely``` library, which GeoPandas relies on for geometric operations. This process involves measuring the shortest distance between the point geometry and the nearest part of the polygon geometry. Specifically, for each point, the distance is calculated to the closest boundary or vertex of the polygon. If a point lies inside a polygon, the distance to the polygon is considered zero.

We can see how useful this may be. For example, calculating distances to environmental hazards or inferring accesibility to green space.  GeoPandas makes this process straightforward through methods like ```.distance()```. When you call this method on a series containing point geometries and pass a polygon geometry as an argument, it returns a new series with the distance from each point to the polygon. Similarly, if applied on a polygon GeoSeries with a point as an argument, it calculates the distances from each polygon to the point. This method is particularly useful in spatial analysis for determining proximity, defining buffer zones, or analyzing spatial relationships between different geometric entities. 

It's important to note that the distance is calculated in the coordinate system of the geometries, so for accurate distance measurements, especially over larger geographic areas, it's advisable to use a coordinate system appropriate for distance measurement, like a projected coordinate system we referred to earlier. As if by magic, we have been using a CRS (EPSG:3857) that is based on metres. The following table outlines many of the EPSG codes that use metres. 'Geographic' means the system uses latitude and longitude degrees, while 'Projected' systems use linear units like meters or feet. In geographic coordinate systems (like EPSG:4326), distances are calculated along the surface of a spheroid, making them more complex to compute, especially over large distances. In contrast, projected systems use a flat, two-dimensional plane, making distance calculations straightforward but potentially less accurate over large areas.


| EPSG Code | Reference System Name                               | Distance Type       |
|-----------|----------------------------------------------------|---------------------|
| 4326      | WGS 84                                              | Geographic          |
| 3857      | WGS 84 / Pseudo-Mercator                            | Projected (Meters)  |
| 32633     | WGS 84 / UTM zone 33N                               | Projected (Meters)  |
| 27700     | OSGB36 / British National Grid                      | Projected (Meters)  |
| 3395      | WGS 84 / World Mercator                             | Projected (Meters)  |
| 3577      | GDA94 / Australian Albers                           | Projected (Meters)  |
| 25832     | ETRS89 / UTM zone 32N                               | Projected (Meters)  |
| 26713     | NAD27 / UTM zone 13N                                | Projected (Meters)  |
| 54004     | World Sinusoidal                                    | Projected (Meters)  |
| 6933      | WGS 84 / NSIDC EASE-Grid 2.0 Global                 | Projected (Meters)  |



## Adding a distance buffer

Our maps already allow us to better understand geosptial variability pollution.  To take this one step further we would like to account for changes in emissions around our air quality stations. We have been working with EPSG:3857 Coordinate Reference System which is based in metres, but how can we interact with this metric? 

The first method we can use is ```.buffer```. The ```.buffer()``` method in GeoPandas is used to create a buffer zone around the geometries in a GeoSeries or GeoDataFrame. This method is particularly useful in spatial analysis, where you might want to create a zone of a specific distance around a point, line, or polygon. More specifically, in our example we can create a 5KM radius around our AURN points and then plot these on the map. The .buffer() method generates a new geometric object that represents an area covered within a given distance from the original geometry. For example, if applied to a point, it will create a circle (or an approximate circle in geographic coordinate systems) around the point. If applied to a line, it creates a strip of specified width along the line.  The distance used in the .buffer() method is in the units of the GeoDataFrame's CRS. For instance, if the CRS is in latitude and longitude (like EPSG:4326), the distance is in degrees, which is not intuitive for distance measurements.

In the following code snippet we use the ```.buffer``` method on the ```AURN_gdf_clipped``` dataframe which creates a new dataframe, ```AURN_zone```. We pass a value of 5000 to represent 1Km. We then preview the new dataframe to confirm we have created a new set of polygons.

In [ ]:
# get circle with 1 km radius 
AURN_zone = AURN_gm_clipped.buffer(1000)
AURN_zone

Now in the following code snippet we once again plot our geometries and AURN points but now we add our new buffer geometries. In this example I specify a strange looking ```color="#ff000033"``` to create a transparent circle so we can better see the 5km distance zone. 

In [ ]:
# Clip PM2.5 raster to Greater Manchester and overlay AURN buffer zones
with rasterio.open(DATA_DIR / 'Pollution/totalpm2_522_2022.tif') as src:
    GMdf_aligned = GMdf_wm.to_crs(src.crs)
    clipped_arr, clip_transform = rio_mask(src, GMdf_aligned.geometry, crop=True)
    crs = src.crs

raster_clipped = clipped_arr[0]
transform = clip_transform

# Mask out zero values
raster_masked_clipped = np.ma.masked_where(raster_clipped <= 0, raster_clipped)

# Apply log10 transform
raster_log_clipped = np.ma.log10(raster_masked_clipped)

fig, ax = plt.subplots(figsize=(10, 10))

GMdf_aligned.plot(color='None', edgecolor='blue', linewidth=1, ax=ax, zorder=4)
cx.add_basemap(ax, crs=crs)

extent = [
    transform[2],
    transform[2] + raster_log_clipped.shape[1] * transform[0],
    transform[5] + raster_log_clipped.shape[0] * transform[4],
    transform[5]
]

im = ax.imshow(raster_log_clipped,
               cmap='gist_rainbow', extent=extent, alpha=0.1, origin='upper',
               norm=colors.Normalize(vmin=np.nanmin(raster_log_clipped),
                                     vmax=np.nanmax(raster_log_clipped)))

# Align AURN buffer zones to the raster CRS and overlay
AURN_zone_aligned = AURN_zone.to_crs(crs)
AURN_zone_aligned.plot(ax=ax, color="#ff000033", edgecolor="black", linewidth=2)

divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="5%", pad=0.05)
cbar = plt.colorbar(im, cax=cax)
cbar.set_label("Log10(PM2.5 Concentration)")

ax.set(title="2022 Log10(PM2.5 Concentration) emissions from NAEI")
plt.xlabel("Easting (m)")
plt.ylabel("Northing (m)")
plt.show()

What if we wanted to understand changes in area emissions to compare with changes from the AURN data? Run the following code and then take a look at the description that follows.

In [ ]:
# Clip the PM2.5 raster to the AURN 1 km buffer zone
with rasterio.open(DATA_DIR / 'Pollution/totalpm2_522_2022.tif') as src:
    zone_arr_raw, _ = rio_mask(src, AURN_zone_aligned.geometry, crop=True)

zone_arr  = zone_arr_raw[0]
zone_mask = np.ma.masked_where(zone_arr <= 0, zone_arr)

# (Optional) log₁₀-space
zone_log = np.ma.log10(zone_mask)

# Compute simple sums
total_conc    = np.nansum(zone_mask)
total_logconc = np.nansum(zone_log)

print(f"Sum of PM₂.₅ conc. in zone (raw units): {total_conc:.2f}")
print(f"Sum of log10(PM₂.₅) in zone:            {total_logconc:.2f}")

##### What Does This Code Do?

This code extracts and analyses **PM₂.₅ air pollution data** within a specific **polygon zone** (the 1 km AURN buffer) from a raster dataset. It clips the data using `rio_mask`, filters out invalid values, applies a logarithmic transformation, and calculates total pollution.

##### Step-by-Step Explanation

##### 1. **Clip the Raster to the AURN Buffer Zone**
```python
with rasterio.open(DATA_DIR / 'Pollution/totalpm2_522_2022.tif') as src:
    zone_arr_raw, _ = rio_mask(src, AURN_zone_aligned.geometry, crop=True)
```
Opens the PM₂.₅ raster and extracts only the pixels that fall within the AURN buffer polygon. `crop=True` trims the output array to the bounding box of the zone.

##### 2. **Extract the Raster Values and Mask Invalid Data**
```python
zone_arr  = zone_arr_raw[0]
zone_mask = np.ma.masked_where(zone_arr <= 0, zone_arr)
```
Selects the first (only) band to get a 2D NumPy array, then masks pixels with values ≤ 0 (no-data or invalid measurements).

##### 3. **(Optional) Apply a Logarithmic Transformation**
```python
zone_log = np.ma.log10(zone_mask)
```
Transforms to log₁₀ space — useful for skewed distributions and easier interpretation.

##### 4. **Calculate Total Pollution in the Zone**
```python
total_conc    = np.nansum(zone_mask)
total_logconc = np.nansum(zone_log)
```
Sums all valid PM₂.₅ values (raw units) and the log-transformed values across the zone.

##### Summary
This approach isolates raster data within any polygon geometry, letting you quantify localised pollution levels — useful in environmental analysis for comparing sites or tracking changes over time.

<div class="alert alert-block alert-success">
<b> Exercise 4: Create a map of PM2.5 emissions in a single ward in Greater Manchester 10 years apart and calculate the difference in totals. <a name="Exercise4"></a>  

</b> In this exercise we repeat the above code, but you will need to take a look at the data provided in the data folder and decide which to use. You will also need to decide whether to use log10 data or leave it in the range provided in the original file.

</div>

<div style="border-left: 6px solid #ff8c00; background-color: #fff3e0; padding: 15px; margin: 15px 0;">

<details>
<summary><strong>🧪 Click to show solution</strong></summary>

```python
stockport = GMdf_wm[GMdf_wm['NAME'] == 'Stockport District (B)']

with rasterio.open(DATA_DIR / 'Pollution/totalpm2_522_2022.tif') as src:
    stockport_aligned = stockport.to_crs(src.crs)
    clipped_arr, clip_transform = rio_mask(src, stockport_aligned.geometry, crop=True)
    crs = src.crs

raster_clipped = clipped_arr[0]
transform = clip_transform
```

</details>

</div>

In [ ]:
#---------- INSERT CODE -------------------
# Select a single ward from the Greater Manchester boundary GeoDataFrame
stockport = GMdf_wm[GMdf_wm['NAME'] == ]

# Open the PM2.5 raster and clip to that ward
with rasterio.open(DATA_DIR / 'Pollution/') as src:
    stockport_aligned = stockport.to_crs(src.crs)
    clipped_arr, clip_transform = rio_mask(src,                          , crop=True)
    crs = src.crs

raster_clipped = clipped_arr[0]
transform = clip_transform
#---------- INSERT CODE -------------------


# Mask out zero values
raster_masked_clipped = np.ma.masked_where(raster_clipped <= 0, raster_clipped)

# Apply log10 transform
raster_log_clipped = np.ma.log10(raster_masked_clipped)

fig, ax = plt.subplots(figsize=(10, 10))

stockport_aligned.plot(color='None', edgecolor='blue', linewidth=1, ax=ax, zorder=4)
cx.add_basemap(ax, crs=crs)

extent = [
    transform[2],
    transform[2] + raster_log_clipped.shape[1] * transform[0],
    transform[5] + raster_log_clipped.shape[0] * transform[4],
    transform[5]
]

im = ax.imshow(raster_log_clipped,
               cmap='gist_rainbow', extent=extent, alpha=0.5, origin='upper',
               norm=colors.Normalize(vmin=np.nanmin(raster_log_clipped),
                                     vmax=np.nanmax(raster_log_clipped)))

divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="5%", pad=0.05)
cbar = plt.colorbar(im, cax=cax)
cbar.set_label("Log10(PM2.5 Concentration)")

ax.set(title="2022 Log10(PM2.5 Concentration) emissions from NAEI")
plt.xlabel("Easting (m)")
plt.ylabel("Northing (m)")
plt.show()

# Pull out the raw 2D array and mask zeros
zone_arr  = clipped_arr[0]
zone_mask = np.ma.masked_where(zone_arr <= 0, zone_arr)
zone_log  = np.ma.log10(zone_mask)

total_conc = np.nansum(zone_mask)
print(f"Sum of PM₂.₅ conc. in zone (raw units): {total_conc:.2f}")

<div style="border-left: 6px solid #1f77b4; background-color: #f0f8ff; padding: 15px; margin: 15px 0;">

### 📚 <span style="color:#1f77b4">Recommended Reading & Documentation</span>

To deepen your understanding of the tools and techniques used in this practical, explore the following official resources and references:

#### 🗺️ Geospatial Data Tools in Python
- 📘 [GeoPandas Documentation](https://geopandas.org/): Guide to working with vector data (points, lines, polygons) using pandas-like syntax.
- 📘 [rioxarray Documentation](https://corteva.github.io/rioxarray/): Tutorial and API reference for loading, clipping, and exporting raster data using `xarray`.
- 📘 [Shapely Documentation](https://shapely.readthedocs.io/): The geometric engine behind many spatial operations (e.g., buffering, distance calculation).
- 📘 [Pyproj Documentation](https://pyproj4.github.io/pyproj/stable/): Information on coordinate reference systems (CRS) and transformations.

#### 🌍 Mapping & Visualisation
- 📘 [Contextily Documentation](https://contextily.readthedocs.io/): Add basemaps (e.g., OpenStreetMap) to geospatial plots with minimal code.
- 📘 [Matplotlib – Geographic Plotting](https://matplotlib.org/stable/gallery/index.html): General plotting tips, including map-friendly formatting.

#### 📊 Data Sources (For Enrichment)
- 📘 [UK Index of Multiple Deprivation (IMD)](https://www.gov.uk/government/statistics/english-indices-of-deprivation-2019): Background and access to social deprivation data.
- 📘 [UK National Atmospheric Emissions Inventory (NAEI)](https://naei.beis.gov.uk/): Explore and download emissions maps used in this and other practicals.

---

These resources are helpful for strengthening your confidence in using spatial data tools and for applying GIS methods to real-world environmental and social challenges.

</div>
